# 03 — Comparing SI, SIR, and SEIR

`laser-init` can emit three model structures. This notebook builds and runs all
three on the same dataset and overlays the infectious curve so the structural
differences are visible:

- **SI** — no recovery; everyone eventually infected (monotonic).
- **SIR** — recovery confers immunity; a single epidemic peak.
- **SEIR** — an exposed/latent period delays and reshapes the peak.

**Note:** this runs three full simulations and is compute-heavy.

In [ ]:
import subprocess
from pathlib import Path

COUNTRY, LEVEL, START, END = "ETH", 2, 2015, 2017
BASE = Path(COUNTRY) / str(START)
if not (BASE / "config.yaml").exists():
    subprocess.run(["laser-init", COUNTRY, str(LEVEL), str(START), str(END)], check=True)

The helper below mirrors the generated model scripts (`si.py`/`sir.py`/`seir.py`)
but returns the total infectious count per tick instead of writing plots.

In [ ]:
import geopandas as gpd
import laser.core.distributions as dists
import numpy as np
import pandas as pd
import yaml
from laser.core import PropertySet
from laser.core.demographics import AliasedDistribution, KaplanMeierEstimator
from laser.generic import SEIR, SI, SIR, Model
from laser.generic.utils import ValuesMap
from laser.generic.vitaldynamics import BirthsByCBR, MortalityByEstimator


def infectious_over_time(model_type, base, nyears=1):
    """Build and run an SI/SIR/SEIR model; return total infectious per tick."""
    base = Path(base)
    cfg = yaml.safe_load((base / "config.yaml").read_text())
    cfg["simulation"]["nyears"] = nyears
    dd = Path(cfg["data_dir"]); df = cfg["datafiles"]
    sc = gpd.read_file(dd / df["shape_data"])
    cxr = pd.read_csv(dd / df["cxr_data"])
    pop = pd.read_csv(dd / df["pop_data"])
    exp = pd.read_csv(dd / df["exp_data"])
    p = PropertySet(cfg["simulation"])
    p += {"nticks": p.nyears * 365, "beta": p.r0 / p.infectious_duration_mean}

    sc["nodeid"] = np.arange(len(sc), dtype=np.int32)
    sc["I"] = 0
    sc.at[int(np.argmax(sc.population)), "I"] = 50
    if model_type in ("SIR", "SEIR"):
        sc["R"] = 0
    if model_type == "SEIR":
        sc["E"] = 0
    sc["S"] = sc.population - sc.I - (sc["R"] if model_type in ("SIR", "SEIR") else 0)

    cbr = cxr.CBR.to_numpy()
    if len(cbr) < p.nyears:
        cbr = np.pad(cbr, (0, p.nyears - len(cbr)), mode="edge")
    br = ValuesMap.from_timeseries(np.repeat(cbr[0 : p.nyears], 365), len(sc))
    model = Model(sc, p, birthrates=br)

    infd = dists.normal(loc=p.infectious_duration_mean, scale=2)
    expd = dists.gamma(shape=p.exposed_duration_shape, scale=p.exposed_duration_scale)
    pyr = AliasedDistribution(pop.PopTotal.to_numpy())
    surv = KaplanMeierEstimator(exp.cumulative_deaths.to_numpy())
    births = BirthsByCBR(model, br, pyr)
    mort = MortalityByEstimator(model, surv)

    if model_type == "SI":
        model.components = [SI.Susceptible(model), SI.Infectious(model),
                            SI.Transmission(model), births, mort]
    elif model_type == "SIR":
        model.components = [SIR.Susceptible(model), SIR.Infectious(model, infd),
                            SIR.Recovered(model), SIR.Transmission(model, infd), births, mort]
    else:
        model.components = [SEIR.Susceptible(model), SEIR.Exposed(model, expd, infd),
                            SEIR.Infectious(model, infd), SEIR.Recovered(model),
                            SEIR.Transmission(model, expd), births, mort]
    model.run()
    return np.asarray(model.nodes.I).sum(axis=1)

In [ ]:
curves = {mt: infectious_over_time(mt, BASE, nyears=1) for mt in ("SI", "SIR", "SEIR")}
{mt: (int(c.max()), int(c.argmax())) for mt, c in curves.items()}  # (peak, peak_day)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for mt, curve in curves.items():
    ax.plot(curve, label=mt, linewidth=2)
ax.set_xlabel("day"); ax.set_ylabel("infectious")
ax.set_title(f"{COUNTRY}: infectious over time by model structure"); ax.legend()
plt.tight_layout()

Expect SI to climb monotonically toward the whole population, while SIR and SEIR
peak and decline as individuals recover; SEIR's incubation period typically delays
and lowers the peak relative to SIR.